# WSMTE — 04_model_training.ipynb
**KAGGLE GPU (T4 x2)** | Trains Configs A–E (30 seeds each from CONFIG['SEEDS']).

Saves results after every run. Saves `config_e_best.keras` for PSO stage.

**Kaggle input dataset required:** `wsmte-processed-data`
- Path: `/kaggle/input/wsmte-processed-data/`
- Contains: X_train/val/test.npy, y_clf/reg_*.npy, class_weights.json

In [ ]:
# ── Cell 1: Install deps + Clone/pull repo + sys.path + CONFIG ─────────────
!pip install keras-tcn --quiet

import os
if os.path.exists('/kaggle/working/WSMTE'):
    !cd /kaggle/working/WSMTE && git pull origin main
else:
    !git clone https://github.com/IAMNEERAJ05/WSMTE.git

import sys
sys.path.insert(0, '/kaggle/working/WSMTE')

from config.config import CONFIG
print(f"Window size    : {CONFIG['window_size']}")
print(f"n_features     : {CONFIG['n_features']}")
print(f"Batch size     : {CONFIG['batch_size']}")
print(f"Max epochs     : {CONFIG['max_epochs']}")
print(f"ES patience    : {CONFIG['early_stopping_patience']}")

## GPU Check

In [ ]:
# ── Cell 2: Verify GPU ────────────────────────────────────────────────────
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print(f'TF version : {tf.__version__}')
print(f'GPUs found : {len(gpus)}')
for g in gpus:
    print(f'  {g}')
if not gpus:
    print('WARNING: No GPU — training will be very slow')

## Load All Data Arrays

In [ ]:
# ── Cell 3: Load processed data from Kaggle dataset ───────────────────────
import numpy as np
import json
import pandas as pd

DATA_DIR = '/kaggle/input/wsmte-processed-data/'

X_train = np.load(DATA_DIR + 'X_train.npy')
X_val   = np.load(DATA_DIR + 'X_val.npy')
X_test  = np.load(DATA_DIR + 'X_test.npy')

y_clf_train = np.load(DATA_DIR + 'y_clf_train.npy')
y_clf_val   = np.load(DATA_DIR + 'y_clf_val.npy')
y_clf_test  = np.load(DATA_DIR + 'y_clf_test.npy')

y_reg_train = np.load(DATA_DIR + 'y_reg_train.npy')
y_reg_val   = np.load(DATA_DIR + 'y_reg_val.npy')
y_reg_test  = np.load(DATA_DIR + 'y_reg_test.npy')

with open(DATA_DIR + 'class_weights.json') as f:
    class_weight = json.load(f)

print(f'X_train : {X_train.shape}  dtype={X_train.dtype}')
print(f'X_val   : {X_val.shape}')
print(f'X_test  : {X_test.shape}')
print(f'y_clf_train : {y_clf_train.shape}  labels={set(y_clf_train.tolist())}')
print(f'y_reg_train : {y_reg_train.shape}')
print(f'class_weight: {class_weight}')

assert X_train.shape[2] == CONFIG['n_features'], \
    f"Expected {CONFIG['n_features']} features, got {X_train.shape[2]}"
print(f'Feature count assertion passed: {X_train.shape[2]} == {CONFIG["n_features"]}')

data = {
    'X_train': X_train, 'X_val': X_val, 'X_test': X_test,
    'y_clf_train': y_clf_train, 'y_clf_val': y_clf_val, 'y_clf_test': y_clf_test,
    'y_reg_train': y_reg_train, 'y_reg_val': y_reg_val, 'y_reg_test': y_reg_test,
    'class_weight': class_weight,
}
print('data dict ready OK')

## Training Loop — Configs A–E
> Each config: 30 seeds from CONFIG['SEEDS']. Results appended to CSV after every run.
> If the session dies, check `ablation_results_partial_AE.csv` and restart from the last incomplete config.

In [ ]:
# ── Cell 4: Import trainer ────────────────────────────────────────────────
from src.training.trainer import train_multi_run

RESULTS_PATH = '/kaggle/working/ablation_results_partial_AE.csv'
print('Trainer imported OK')
print(f'Results will be saved to: {RESULTS_PATH}')
print(f'Seeds ({len(CONFIG["SEEDS"])}): {CONFIG["SEEDS"]}')

In [ ]:
# ── Cell 5: Run Config A (raw OHLCV denoised only — price-only floor) ─────
cfg_a = CONFIG['ablation_configs']['A']
print(f"Config A: {cfg_a['description']}")
print(f"Features ({len(cfg_a['features'])}): {cfg_a['features']}")
print(f"n_runs: {cfg_a['n_runs']}  seeds: {CONFIG['SEEDS'][:cfg_a['n_runs']]}")

results_A = train_multi_run(
    ablation_cfg=cfg_a,
    config_name='A',
    data=data,
    config=CONFIG,
    n_runs=cfg_a['n_runs'],
    results_path=RESULTS_PATH,
)
print(f'\nConfig A done. Mean accuracy: {results_A["accuracy"].mean():.4f}')
print(results_A[['run','seed','accuracy','auc','f1']].to_string(index=False))

In [ ]:
# ── Cell 6: Run Config B (denoised OHLCV + all sentiment) ─────────────────
cfg_b = CONFIG['ablation_configs']['B']
print(f"Config B: {cfg_b['description']}")
print(f"Features ({len(cfg_b['features'])}): {cfg_b['features']}")

results_B = train_multi_run(
    ablation_cfg=cfg_b,
    config_name='B',
    data=data,
    config=CONFIG,
    n_runs=cfg_b['n_runs'],
    results_path=RESULTS_PATH,
)
print(f'\nConfig B done. Mean accuracy: {results_B["accuracy"].mean():.4f}')
print(results_B[['run','seed','accuracy','auc','f1']].to_string(index=False))

In [ ]:
# ── Cell 7: Run Config C (technical indicators only) ──────────────────────
cfg_c = CONFIG['ablation_configs']['C']
print(f"Config C: {cfg_c['description']}")
print(f"Features ({len(cfg_c['features'])}): {cfg_c['features']}")

results_C = train_multi_run(
    ablation_cfg=cfg_c,
    config_name='C',
    data=data,
    config=CONFIG,
    n_runs=cfg_c['n_runs'],
    results_path=RESULTS_PATH,
)
print(f'\nConfig C done. Mean accuracy: {results_C["accuracy"].mean():.4f}')
print(results_C[['run','seed','accuracy','auc','f1']].to_string(index=False))

In [ ]:
# ── Cell 8: Run Config D (all features, single-task — full feature set) ────
cfg_d = CONFIG['ablation_configs']['D']
print(f"Config D: {cfg_d['description']}")
print(f"Features ({len(cfg_d['features'])}): {cfg_d['features']}")

results_D = train_multi_run(
    ablation_cfg=cfg_d,
    config_name='D',
    data=data,
    config=CONFIG,
    n_runs=cfg_d['n_runs'],
    results_path=RESULTS_PATH,
)
print(f'\nConfig D done. Mean accuracy: {results_D["accuracy"].mean():.4f}')
print(results_D[['run','seed','accuracy','auc','f1']].to_string(index=False))

In [ ]:
# ── Cell 9: Run Config E (all features, both heads, concat — full WSMTE without PSO)
cfg_e = CONFIG['ablation_configs']['E']
print(f"Config E: {cfg_e['description']}")
print(f"Features ({len(cfg_e['features'])}): {cfg_e['features']}")
print(f"Heads: {cfg_e['heads']}  Merge: {cfg_e['merge']}")

results_E = train_multi_run(
    ablation_cfg=cfg_e,
    config_name='E',
    data=data,
    config=CONFIG,
    n_runs=cfg_e['n_runs'],
    results_path=RESULTS_PATH,
)
print(f'\nConfig E done. Mean accuracy: {results_E["accuracy"].mean():.4f}')
print(results_E[['run','seed','accuracy','balanced_accuracy','auc','f1','rmse']].to_string(index=False))

## Save Config E Best Model Weights
> Re-train with the best seed from Config E results to get a model object for the PSO stage.

In [ ]:
# ── Cell 10: Retrain best Config E seed to get model for PSO stage ────────
from src.models.wsmte import build_wsmte
from src.training.callbacks import get_callbacks

best_seed_e = int(results_E.loc[results_E['accuracy'].idxmax(), 'seed'])
print(f'Best Config E seed : {best_seed_e}')
print(f'Best Config E acc  : {results_E["accuracy"].max():.4f}')

tf.random.set_seed(best_seed_e)
np.random.seed(best_seed_e)

feat_idx = [CONFIG['feature_columns'].index(f) for f in cfg_e['features']]
X_tr = X_train[:, :, feat_idx]
X_vl = X_val[:, :, feat_idx]

e_model = build_wsmte(CONFIG, ablation_cfg=cfg_e)
e_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=CONFIG['learning_rate'])
)
e_model.fit(
    X_tr, [data['y_reg_train'], data['y_clf_train']],
    validation_data=(X_vl, [data['y_reg_val'], data['y_clf_val']]),
    epochs=CONFIG['max_epochs'],
    batch_size=CONFIG['batch_size'],
    callbacks=get_callbacks(CONFIG, run_id='config_E_best'),
    verbose=1,
)
print('Config E best model retrained OK')

In [ ]:
# ── Cell 11: Save Config E best model ─────────────────────────────────────
e_model.save('/kaggle/working/config_e_best.keras')
print('Saved -> /kaggle/working/config_e_best.keras')
print(f'Trainable params: {e_model.count_params():,}')

## Save All Results

In [ ]:
# ── Cell 12: Combine A–E and save final results CSV ───────────────────────
all_results = pd.concat(
    [results_A, results_B, results_C, results_D, results_E],
    ignore_index=True
)
all_results.to_csv('/kaggle/working/ablation_results_AE.csv', index=False)
print(f'Saved -> /kaggle/working/ablation_results_AE.csv')
print(f'Shape : {all_results.shape}')
print()
summary = all_results.groupby('config')['accuracy'].agg(['mean', 'max', 'std'])
summary.columns = ['mean_acc', 'max_acc', 'std_acc']
print(summary.round(4).to_string())

## Download Instructions

In [ ]:
# ── Cell 13: Remind what to download ──────────────────────────────────────
import os
downloads = [
    '/kaggle/working/ablation_results_AE.csv',
    '/kaggle/working/config_e_best.keras',
]
print('Files to download from Kaggle Output:')
for p in downloads:
    size = os.path.getsize(p) // 1024 if os.path.exists(p) else 'MISSING'
    print(f'  {p}  ({size} KB)')
print()
print('Local destinations:')
print('  ablation_results_AE.csv  -> ablation/')
print('  config_e_best.keras      -> results/saved_models/')
print()
print('Expected Kaggle runtime: 4–8 hours (5 configs × 30 seeds)')